# Pipeline Evaluation

Evaluates the perception pipeline against RoboFail ground truth across all episodes.

| Metric | Stage | Source |
|---|---|---|
| Detection Recall | 1 | GT object list vs detected labels |
| Localization Flag Accuracy | 5 | GT failure type vs pipeline flag |
| Slip / No_Grasp / Translation breakdown | 5 | Per-episode flag types |
| ID Switch Rate | 4 | `track/*.npz` id_switch_frames |
| Relation type distribution | 5 | `scene_graphs_pipeline/*.json` |
| Held_by_gripper false-positive rate | 4+5 | Large-object gate check |

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

ROOT = Path("../")
ALIGNED_DIR     = ROOT / "aligned"
DETECT_DIR      = ROOT / "detect"
TRACK_DIR       = ROOT / "track"
GRAPHS_DIR      = ROOT / "scene_graphs_pipeline"
DATA_DIR        = ROOT / "data"

plt.rcParams.update({"figure.dpi": 130, "font.size": 10})
print("ROOT:", ROOT.resolve())

## 1 — Load all episode data

In [ ]:
def load_gt_failure(episode_id: str) -> dict:
    """Load ground-truth failure info from task.json."""
    for task_dir in DATA_DIR.iterdir():
        if not task_dir.is_dir():
            continue
        for ep_dir in task_dir.iterdir():
            if ep_dir.name == episode_id:
                task_json = ep_dir / "task.json"
                if task_json.exists():
                    with open(task_json) as f:
                        meta = json.load(f)
                    return {
                        "gt_reason": meta.get("gt_failure_reason", "Unknown"),
                        "gt_step":   meta.get("gt_failure_step", None),
                        "object_list": meta.get("object_list", []),
                    }
    # real-world
    rw = DATA_DIR / "tasks_real_world.json"
    if rw.exists():
        with open(rw) as f:
            real = json.load(f)
        for key, ep in real.items():
            if key == episode_id:
                return {
                    "gt_reason": ep.get("gt_failure_reason", "Unknown"),
                    "gt_step":   ep.get("gt_failure_step", None),
                    "object_list": ep.get("object_list", []),
                }
    return {"gt_reason": "Unknown", "gt_step": None, "object_list": []}


# Rough mapping from GT failure reason strings to taxonomy types
GT_TO_TYPE = {
    "dropped": "Slip",
    "slip":    "Slip",
    "wrong":   "Wrong_object",
    "missing": "Wrong_object",
    "grasp":   "No_Grasp",
    "no grasp":"No_Grasp",
    "translat":"Translation",
    "moved":   "Translation",
}

def gt_reason_to_type(reason: str) -> str:
    r = reason.lower()
    for key, typ in GT_TO_TYPE.items():
        if key in r:
            return typ
    return "Unknown"


# Collect all available episodes with pipeline output
episodes = sorted(p.stem for p in GRAPHS_DIR.glob("*.json"))
print(f"Episodes with pipeline output: {len(episodes)}")
print(episodes)

In [ ]:
records = []

for ep in episodes:
    graph_path  = GRAPHS_DIR / f"{ep}.json"
    detect_path = DETECT_DIR / f"{ep}.npz"
    track_path  = TRACK_DIR  / f"{ep}.npz"

    with open(graph_path) as f:
        gdata = json.load(f)

    frames = gdata["frames"]
    N = len(frames)

    # ground truth
    gt = load_gt_failure(ep)
    gt_type = gt_reason_to_type(gt["gt_reason"])

    # pipeline localization flag for failure frames
    fail_frames = [fr for fr in frames if fr["failure_label"]]
    pipeline_type = None
    if fail_frames:
        flag = fail_frames[0]["localization_flag"]
        pipeline_type = flag.get("type") if flag.get("failure_detected") else None

    # detection recall: fraction of GT objects detected (by label) in failure frames
    gt_labels = set(l.lower() for l in gt["object_list"])
    detected_labels: set[str] = set()
    if detect_path.exists():
        det = np.load(detect_path, allow_pickle=True)
        vocab = list(det["label_vocab"])
        label_ids = det["label_ids"]   # (N, MAX_DET)
        n_dets    = det["n_dets"]      # (N,)
        for i, fr in enumerate(frames):
            if fr["failure_label"] and i < len(n_dets):
                for j in range(int(n_dets[i])):
                    lid = int(label_ids[i, j])
                    if 0 <= lid < len(vocab):
                        detected_labels.add(vocab[lid].lower())
    recall = (len(gt_labels & detected_labels) / len(gt_labels)
              if gt_labels else float("nan"))

    # ID switch rate
    n_switches = 0
    if track_path.exists():
        trk = np.load(track_path, allow_pickle=True)
        n_switches = len(trk["id_switch_frames"])

    # relation stats from all frames
    all_relations = [r for fr in frames for r in fr["spatial_relations"]]
    rel_counts: dict[str, int] = {}
    for r in all_relations:
        rel_counts[r["relation"]] = rel_counts.get(r["relation"], 0) + 1

    # held_by_gripper stats
    n_held = sum(
        1 for fr in frames for o in fr["objects"] if o["held_by_gripper"]
    )

    records.append({
        "episode":       ep,
        "n_frames":      N,
        "n_fail_frames": len(fail_frames),
        "gt_reason":     gt["gt_reason"],
        "gt_type":       gt_type,
        "pipeline_type": pipeline_type,
        "correct":       gt_type == pipeline_type,
        "recall":        recall,
        "n_switches":    n_switches,
        "rel_counts":    rel_counts,
        "n_held":        n_held,
    })

print(f"Loaded {len(records)} episode records")

## 2 — Localization Flag Accuracy

In [ ]:
n_correct = sum(1 for r in records if r["correct"])
n_total   = len(records)
acc = n_correct / n_total if n_total else 0

print(f"Localization accuracy: {n_correct}/{n_total} = {acc:.1%}")
print()
print(f"{'Episode':<22} {'GT type':<16} {'Pipeline':<16} {'Match'}")
print("-" * 65)
for r in records:
    tick = "✓" if r["correct"] else "✗"
    print(f"{r['episode']:<22} {r['gt_type']:<16} {str(r['pipeline_type']):<16} {tick}")

In [ ]:
# Confusion matrix
all_types = sorted(set(
    list({r["gt_type"] for r in records}) +
    list({str(r["pipeline_type"]) for r in records})
))
idx = {t: i for i, t in enumerate(all_types)}
mat = np.zeros((len(all_types), len(all_types)), dtype=int)

for r in records:
    gt_i   = idx[r["gt_type"]]
    pred_i = idx[str(r["pipeline_type"])]
    mat[gt_i, pred_i] += 1

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(mat, cmap="Blues")
ax.set_xticks(range(len(all_types))); ax.set_xticklabels(all_types, rotation=30, ha="right")
ax.set_yticks(range(len(all_types))); ax.set_yticklabels(all_types)
ax.set_xlabel("Predicted"); ax.set_ylabel("Ground Truth")
ax.set_title(f"Localization Confusion Matrix  (acc={acc:.1%})")
for i in range(len(all_types)):
    for j in range(len(all_types)):
        if mat[i, j] > 0:
            ax.text(j, i, str(mat[i, j]), ha="center", va="center",
                    color="white" if mat[i, j] > mat.max() * 0.5 else "black")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig(ROOT / "analysis" / "localization_confusion.png", dpi=150)
plt.show()

## 3 — Detection Recall

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ep_labels = [r["episode"] for r in records]
recalls   = [r["recall"] if not np.isnan(r["recall"]) else 0.0 for r in records]
colors    = ["#e74c3c" if rec < 0.5 else "#2ecc71" if rec >= 1.0 else "#f39c12"
             for rec in recalls]
bars = ax.bar(ep_labels, recalls, color=colors, edgecolor="white", linewidth=0.5)
ax.axhline(1.0, color="#27ae60", linestyle="--", alpha=0.6, label="perfect recall")
ax.set_ylim(0, 1.15)
ax.set_ylabel("Recall (GT objects detected at failure frame)")
ax.set_title("Stage 1 — Detection Recall per Episode")
ax.tick_params(axis="x", rotation=40)
for bar, val in zip(bars, recalls):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f"{val:.0%}", ha="center", va="bottom", fontsize=8)
patches = [
    mpatches.Patch(color="#2ecc71", label="100% recall"),
    mpatches.Patch(color="#f39c12", label="partial recall"),
    mpatches.Patch(color="#e74c3c", label="<50% recall"),
]
ax.legend(handles=patches, loc="upper right")
plt.tight_layout()
plt.savefig(ROOT / "analysis" / "detection_recall.png", dpi=150)
plt.show()
print(f"Mean recall: {np.nanmean(recalls):.1%}")

## 4 — ID Switch Rate (Stage 4 Tracker)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# raw switch counts
n_sw    = [r["n_switches"] for r in records]
n_fr    = [r["n_frames"]   for r in records]
sw_rate = [s / f if f else 0 for s, f in zip(n_sw, n_fr)]

ax1.bar(ep_labels, n_sw, color="#3498db", edgecolor="white")
ax1.set_title("ID Switches (raw count)")
ax1.set_ylabel("# switch frames")
ax1.tick_params(axis="x", rotation=40)

ax2.bar(ep_labels, sw_rate, color="#9b59b6", edgecolor="white")
ax2.set_title("ID Switch Rate (switches / frames)")
ax2.set_ylabel("Rate")
ax2.tick_params(axis="x", rotation=40)

plt.suptitle("Stage 4 — Temporal Tracking Stability")
plt.tight_layout()
plt.savefig(ROOT / "analysis" / "id_switch_rate.png", dpi=150)
plt.show()
print(f"Mean switch rate: {np.mean(sw_rate):.3f} switches/frame")

## 5 — Spatial Relation Distribution (Stage 5)

In [ ]:
# Aggregate across all episodes
total_rel: dict[str, int] = {}
for r in records:
    for rel, cnt in r["rel_counts"].items():
        total_rel[rel] = total_rel.get(rel, 0) + cnt

rel_types  = sorted(total_rel.keys())
rel_counts = [total_rel[t] for t in rel_types]
pal = ["#e74c3c", "#e67e22", "#f1c40f", "#2ecc71", "#3498db", "#9b59b6"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.bar(rel_types, rel_counts, color=pal[:len(rel_types)], edgecolor="white")
ax1.set_title("Total Spatial Relations by Type (all episodes)")
ax1.set_ylabel("Count")
ax1.tick_params(axis="x", rotation=20)

ax2.pie(rel_counts, labels=rel_types, colors=pal[:len(rel_types)],
        autopct="%1.0f%%", startangle=140)
ax2.set_title("Relation Type Distribution")

plt.suptitle("Stage 5 — Scene Graph Relation Composition")
plt.tight_layout()
plt.savefig(ROOT / "analysis" / "relation_distribution.png", dpi=150)
plt.show()
print("Relation totals:", total_rel)

## 6 — Per-Episode Relations vs Failure Frame

In [ ]:
# For each episode plot relation count per frame, highlighting failure frames
n_eps = len(episodes)
ncols = min(3, n_eps)
nrows = (n_eps + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3 * nrows))
axes = np.array(axes).flatten()

for ax, ep in zip(axes, episodes):
    graph_path = GRAPHS_DIR / f"{ep}.json"
    if not graph_path.exists():
        ax.set_visible(False)
        continue
    with open(graph_path) as f:
        gdata = json.load(f)
    frames = gdata["frames"]
    n_rels  = [len(fr["spatial_relations"]) for fr in frames]
    is_fail = [fr["failure_label"] for fr in frames]
    x = list(range(len(frames)))
    colors_fr = ["#e74c3c" if f else "#3498db" for f in is_fail]
    ax.bar(x, n_rels, color=colors_fr, width=0.8)
    ax.set_title(ep, fontsize=9)
    ax.set_xlabel("Frame"); ax.set_ylabel("# Relations")
    ax.set_xlim(-0.5, len(frames) - 0.5)

for ax in axes[n_eps:]:
    ax.set_visible(False)

legend_patches = [
    mpatches.Patch(color="#e74c3c", label="failure frame"),
    mpatches.Patch(color="#3498db", label="normal frame"),
]
fig.legend(handles=legend_patches, loc="lower right")
plt.suptitle("Spatial Relation Count per Frame", y=1.01)
plt.tight_layout()
plt.savefig(ROOT / "analysis" / "relations_per_frame.png", dpi=150, bbox_inches="tight")
plt.show()

## 7 — Held-by-Gripper Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
held_counts = [r["n_held"] for r in records]
ax.bar(ep_labels, held_counts, color="#1abc9c", edgecolor="white")
ax.set_title("held_by_gripper Detections per Episode")
ax.set_ylabel("Total held-object frames")
ax.tick_params(axis="x", rotation=40)
for bar, val in zip(ax.patches, held_counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
            str(val), ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.savefig(ROOT / "analysis" / "held_by_gripper.png", dpi=150)
plt.show()

## 8 — Summary Table

In [ ]:
print(f"{'Episode':<22} {'GT type':<16} {'Pred type':<16} {'Recall':>7} {'ID switches':>12} {'Held det':>9}")
print("-" * 88)
for r in records:
    tick = "✓" if r["correct"] else "✗"
    rec  = f"{r['recall']:.0%}" if not np.isnan(r["recall"]) else "N/A"
    print(
        f"{r['episode']:<22} {r['gt_type']:<16} "
        f"{str(r['pipeline_type']):<14} {tick}  {rec:>7} "
        f"{r['n_switches']:>12}  {r['n_held']:>7}"
    )
print("-" * 88)
mean_rec = np.nanmean([r["recall"] for r in records])
print(f"{'MEAN / TOTAL':<22} {'':16} {'':16} {mean_rec:>7.0%} "
      f"{sum(r['n_switches'] for r in records):>12}  "
      f"{sum(r['n_held'] for r in records):>7}")
print(f"\nLocalization accuracy: {n_correct}/{n_total} = {acc:.1%}")